# TODO
Transfer daset from 1/12 to 1/4

- 1/4 daset oisst v2.1 :'/lustre/home/yuhanxue/data/oisst/V2.1/oisst-avhrr-v02r01.19810901.nc'
- 1/12 daset glorys s1v2 :'/lustre/home/yuhanxue/data/GLORYS/Area1/thetao&uo&vo_time1993-01-01~1993-01-31_LonMin195_LonMax230_LatMin35_LatMax50_DMax318.1274108886719.nc'

In [1]:
import os
import numpy as np
import scipy as sp
import pandas as pd
from datetime import date
import marineHeatWaves as mhw
import netCDF4 as nc
import datetime
import matplotlib.pyplot as plt 
from tqdm import notebook
from scipy.interpolate import griddata
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, Executor

In [2]:
lat_4=np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/lats.npy')
lon_4=np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/lons.npy')-360

In [3]:
print(f'lat_4 range:{lat_4[0]:.2f}~{lat_4[-1]:.2f} | freq:{abs(lat_4[1]-lat_4[0])}\nlon_4 range:{lon_4[0]:.2f}~{lon_4[-1]:.2f} | freq:{lon_4[1]-lon_4[0]}')

lat_4 range:50.00~20.00 | freq:0.25
lon_4 range:-195.00~-130.00 | freq:0.25


In [4]:
lat_12=np.load('/lustre/home/yuhanxue/data/GLORYS/area1_re/lat.npy')
lon_12=np.load('/lustre/home/yuhanxue/data/GLORYS/area1_re/lon.npy')

In [5]:
print(f'lat_12 range:{lat_12[0]:.2f}~{lat_12[-1]:.2f} | freq:{lat_12[1]-lat_12[0]}\nlon_12 range:{lon_12[0]:.2f}~{lon_12[-1]:.2f} | freq:{lon_12[1]-lon_12[0]}')

lat_12 range:35.00~50.00 | freq:0.08333206176757812
lon_12 range:-165.00~-130.00 | freq:0.0833282470703125


经上述数据观察需要将ERA/0.25数据集进行如下操作：  
* axis=lat 反转  
* axis=lat 缩小至35~50  
* axis=lon 缩小至-165~-130(165W~130W)  

需要将Glorys数据进行以下操作
* 加载数据  
* 生成插值网点  
* 二维插值  

## ERA/0.25 数据集操作

In [14]:
u10s=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/u10s.npy')
v10s=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/v10s.npy')
hccs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/hccs.npy')
lccs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/lccs.npy')
msls=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/msls.npy')
mccs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/mccs.npy')
slhfs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/slhfs.npy')
ssrs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/ssrs.npy')
strs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/strs.npy')
sshfs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/sshfs.npy')

### axis=lat 反转

In [28]:
lat_4=lat_4[::-1]
u10s=u10s[:,::-1,:]
v10s=v10s[:,::-1,:]
hccs=hccs[:,::-1,:]
lccs=lccs[:,::-1,:]
msls=msls[:,::-1,:]
mccs=mccs[:,::-1,:]
slhfs=slhfs[:,::-1,:]
ssrs=ssrs[:,::-1,:]
strs=strs[:,::-1,:]
sshfs=sshfs[:,::-1,:]

### axis=lat 缩小至35~50 & axis=lon 缩小至-165~-130(165W~130W)

In [54]:
lat_4_ind=(lat_4>=35)&(lat_4<=50)
lon_4_ind=(lon_4>=-165)&(lon_4<=-130)
u10s=u10s[:,lat_4_ind,:][:,:,lon_4_ind]
v10s=v10s[:,lat_4_ind,:][:,:,lon_4_ind]
hccs=hccs[:,lat_4_ind,:][:,:,lon_4_ind]
lccs=lccs[:,lat_4_ind,:][:,:,lon_4_ind]
msls=msls[:,lat_4_ind,:][:,:,lon_4_ind]
mccs=mccs[:,lat_4_ind,:][:,:,lon_4_ind]
slhfs=slhfs[:,lat_4_ind,:][:,:,lon_4_ind]
ssrs=ssrs[:,lat_4_ind,:][:,:,lon_4_ind]
strs=strs[:,lat_4_ind,:][:,:,lon_4_ind]
sshfs=sshfs[:,lat_4_ind,:][:,:,lon_4_ind]

### 结束处理，数据回存

In [55]:
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/u10s.npy',u10s)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/v10s.npy',v10s)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/hccs.npy',hccs)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/lccs.npy',lccs)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/msls.npy',msls)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/mccs.npy',mccs)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/slhfs.npy',slhfs)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/ssrs.npy',ssrs)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/strs.npy',strs)
np.save(f'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/sshfs.npy',sshfs)

## GLORYS数据处理

In [7]:
lat_4_ind=(lat_4>=35)&(lat_4<=50)
lon_4_ind=(lon_4>=-165)&(lon_4<=-130)
Lon_4,Lat_4=np.meshgrid(lon_4[lon_4_ind],lat_4[lat_4_ind])
Lon_12,Lat_12=np.meshgrid(lon_12,lat_12)
points=np.array([Lon_12.flatten(),Lat_12.flatten()]).T
def grid(dat):
    global points,Lon_4,Lat_4
    return griddata(points,dat.flatten(),(Lon_4,Lat_4),'cubic')
def list_map(dat):
    pool = ProcessPoolExecutor(max_workers=9)
    ans=np.array(list(pool.map(grid,dat)))
    del pool
    return ans

In [8]:
pool = ProcessPoolExecutor(max_workers=7)
thetaos=np.load('/lustre/home/yuhanxue/data/GLORYS/area1_re/thetaos.npy')
thetaos_new=np.array(list(pool.map(list_map,thetaos[:,:,:,:])))
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/thetaos_new.npy',thetaos_new)
del thetaos,thetaos_new
print('thetaos')
uos=np.load('/lustre/home/yuhanxue/data/GLORYS/area1_re/uos.npy')
uos_new=np.array(list(pool.map(list_map,uos[:,:,:,:])))
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/uos_new.npy',uos_new)
del uos,uos_new
print('uos')
vos=np.load('/lustre/home/yuhanxue/data/GLORYS/area1_re/vos.npy')
vos_new=np.array(list(pool.map(list_map,vos[:,:,:,:])))
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/vos_new.npy',vos_new)
del vos,vos_new
print('uos')

thetaos
uos
uos


In [ ]:
np.save('/lustre/home/yuhanxue/data/GLORYS/area1_re/uos_new.npy',uos_new)

In [11]:
np.save('12_2_mld_lon.npy',lon_4[lon_4_ind])

In [12]:
np.save('12_2_mld_lat.npy',lat_4[lat_4_ind])

In [14]:
ls /lustre/home/yuhanxue/data/ERA/0.25area/re

hccs.npy  lccs.npy  mccs.npy  new/       sshfs.npy  strs.npy    u10s.npy
lats.npy  lons.npy  msls.npy  slhfs.npy  ssrs.npy   timess.npy  v10s.npy


In [20]:
times=pd.to_datetime(np.load('/lustre/home/yuhanxue/data/ERA/0.25area/re/timess.npy',allow_pickle=True))

In [26]:
times[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]

DatetimeIndex(['2021-07-01', '2021-07-02', '2021-07-03', '2021-07-04',
               '2021-07-05', '2021-07-06', '2021-07-07', '2021-07-08',
               '2021-07-09', '2021-07-10', '2021-07-11', '2021-07-12',
               '2021-07-13', '2021-07-14', '2021-07-15', '2021-07-16',
               '2021-07-17', '2021-07-18', '2021-07-19', '2021-07-20',
               '2021-07-21', '2021-07-22', '2021-07-23', '2021-07-24',
               '2021-07-25', '2021-07-26', '2021-07-27', '2021-07-28',
               '2021-07-29', '2021-07-30', '2021-07-31', '2021-08-01',
               '2021-08-02', '2021-08-03', '2021-08-04', '2021-08-05',
               '2021-08-06', '2021-08-07', '2021-08-08', '2021-08-09',
               '2021-08-10', '2021-08-11', '2021-08-12', '2021-08-13',
               '2021-08-14', '2021-08-15', '2021-08-16', '2021-08-17',
               '2021-08-18', '2021-08-19', '2021-08-20', '2021-08-21',
               '2021-08-22', '2021-08-23', '2021-08-24', '2021-08-25',
      

In [27]:
u10s=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/u10s.npy')
v10s=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/v10s.npy')
hccs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/hccs.npy')
lccs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/lccs.npy')
msls=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/msls.npy')
mccs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/mccs.npy')
slhfs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/slhfs.npy')
ssrs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/ssrs.npy')
strs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/strs.npy')
sshfs=np.load(rf'/lustre/home/yuhanxue/data/ERA/0.25area/re/new/sshfs.npy')

In [28]:
u10s_2021_789=u10s[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
v10s_2021_789=v10s[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
hccs_2021_789=hccs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
lccs_2021_789=lccs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
msls_2021_789=msls[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
mccs_2021_789=mccs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
slhfs_2021_789=slhfs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
ssrs_2021_789=ssrs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
strs_2021_789=strs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]
sshfs_2021_789=sshfs[(times.year==2021)&((times.month==7)|(times.month==8)|(times.month==9))]

In [29]:
np.savez('12_2_Q_2021_789.npz',slhf=slhfs_2021_789,ssr=ssrs_2021_789,str=strs_2021_789,sshf=sshfs_2021_789)

In [30]:
import scipy.io as scio
scio.savemat('12_2_Q_2021_789.mat',{'slhf':slhfs_2021_789,'ssr':ssrs_2021_789,'str':strs_2021_789,'sshf':sshfs_2021_789})